In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
from collections import defaultdict
from genomic_benchmarks.dataset_getters.pytorch_datasets import get_dataset
# Import the genomic benchmark dataset from Hugging Face
from genomic_benchmarks.dataset_getters.pytorch_datasets import DemoMouseEnhancers

# --------------------
# Data Preprocessing
# --------------------




    # Load the genomic benchmark dataset from Hugging Face
train_dset = get_dataset("demo_coding_vs_intergenomic_seqs", split="train", version=0)
test_dset  = get_dataset("demo_coding_vs_intergenomic_seqs", split="test", version=0)

    # Extract sequences and labels from the datasets
train_seqs = [item[0] for item in train_dset]
train_labels = [item[1] for item in train_dset]
test_seqs = [item[0] for item in test_dset]
test_labels = [item[1] for item in test_dset]

    



C:\Users\FA007\.conda\envs\new2\Lib\site-packages\genomic_benchmarks\utils\datasets.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
from genomic_benchmarks.data_check import list_datasets
print(list_datasets())

['human_ensembl_regulatory', 'human_ocr_ensembl', 'demo_human_or_worm', 'demo_coding_vs_intergenomic_seqs', 'dummy_mouse_enhancers_ensembl', 'human_nontata_promoters', 'drosophila_enhancers_stark', 'human_enhancers_ensembl', 'human_enhancers_cohn']


In [3]:
import pandas as pd
# converting the lists to train and test csv
train_df = pd.DataFrame({
    'sequence':train_seqs,
    'target': train_labels
})

In [4]:
test_df = pd.DataFrame({
    'sequence':test_seqs,
    'target': test_labels
})

In [5]:
train_df.head(5)

,sequence,target
0,TTTTTCAGAAGAGCAGAATGTTTATTTGCCAAGACCTGTTTGGCAG...,0
1,GGATGGCGAGCAGCGGAGGCGAGGCGGTGACGAGAGCAGCGGCTCC...,0
2,GCCTGCGCCGCGTCGGCGTGCGGAACGCCGCGGTGTCTCGGCGCCT...,0
3,TTGGCAGGAAGAAGGGAGAGAAATCGGGGGGCTGCCCGCCAGATGA...,0
4,TATCAAGTCATGATCCGCTGTGGAGAAGACATTGCAAAAAATACTG...,0


In [6]:
train_df.shape

(75000, 2)

In [7]:
test_df.head(5)

,sequence,target
0,GAGCCTTCACCCACCGTAATGAGCACGTCCCTGGGCAGCAACCTTT...,0
1,TTCTTGTGACCCTGGTGGCTGTGGAAGGAGTTAAAGAGGGTATAGA...,0
2,TCACAGGGTCCAACACTGCCCCAGGAAGTTTTCTGTTTTCTTCCTG...,0
3,TTTTTAGGAGGGGTTTAAAAGTTTGATGGGGTAGAAGTAAACGTTG...,0
4,CAGACACACCGAAGCCCTACTCAGTCACAACCAGCTTTCTTGGCCA...,0


In [8]:
test_df.shape

(25000, 2)

In [9]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("InstaDeepAI/nucleotide-transformer-v2-100m-multi-species", trust_remote_code=True)

# Load model using EsmModel
model = AutoModelForMaskedLM.from_pretrained("InstaDeepAI/nucleotide-transformer-v2-100m-multi-species", trust_remote_code=True).to(device)

def embed_sequence(seq):
    inputs = tokenizer(seq, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs[0].mean(dim=1).squeeze().cpu().numpy() 
import time

# Start timer
start_time = time.time()
# Generate embeddings for train/test sequences
train_df["embedding"] = train_df["sequence"].apply(embed_sequence)
test_df["embedding"] = test_df["sequence"].apply(embed_sequence)
import faiss
import numpy as np

# Convert embeddings to numpy arrays
train_embeddings = np.stack(train_df["embedding"].values)
test_embeddings = np.stack(test_df["embedding"].values)

# Build FAISS index
dim = train_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(train_embeddings)
k = 5  # Number of neighbors to retrieve

# Retrieve nearest neighbors for test data
_, indices = index.search(test_embeddings, k)

# Get labels of retrieved training examples
retrieved_labels = train_df.iloc[indices.flatten()]["target"].values.reshape(-1, k)
# Example: Majority voting
from scipy.stats import mode

predicted_labels, _ = mode(retrieved_labels, axis=1)
predicted_labels = predicted_labels.squeeze()
from sklearn.metrics import accuracy_score, f1_score, classification_report

true_labels = test_df["target"].values
print(f"Accuracy: {accuracy_score(true_labels, predicted_labels)}")
print(classification_report(true_labels, predicted_labels))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

Accuracy: 0.85404
              precision    recall  f1-score   support

           0       0.90      0.80      0.85     12500
           1       0.82      0.91      0.86     12500

    accuracy                           0.85     25000
   macro avg       0.86      0.85      0.85     25000
weighted avg       0.86      0.85      0.85     25000


🕒 Evaluation took 3426.74 seconds (57.11 minutes)


In [10]:
import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)
import time

# Start timer
start_time = time.time()
# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)
    # Add other functions as needed.
}

# Choose the features you want to try. For example:
# To try a combination of gcContent + zCurve:
selected_feature_keys = [ 'gcContent']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(np.stack(train_df["embedding"].values))
test_embeddings = normalize_embeddings(np.stack(test_df["embedding"].values))

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

              precision    recall  f1-score   support

           0       0.92      0.78      0.84     12500
           1       0.81      0.93      0.86     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.85     25000
weighted avg       0.86      0.86      0.85     25000


🕒 Evaluation took 51.25 seconds (0.85 minutes)


In [11]:
import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)
import time

# Start timer
start_time = time.time()
# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)

}

# Choose the features you want to try.

selected_feature_keys = [ 'zCurve']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(np.stack(train_df["embedding"].values))
test_embeddings = normalize_embeddings(np.stack(test_df["embedding"].values))

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

              precision    recall  f1-score   support

           0       0.92      0.78      0.84     12500
           1       0.81      0.93      0.86     12500

    accuracy                           0.85     25000
   macro avg       0.86      0.85      0.85     25000
weighted avg       0.86      0.85      0.85     25000


🕒 Evaluation took 50.83 seconds (0.85 minutes)


In [12]:
import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)
import time

# Start timer
start_time = time.time()
# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)

}

# Choose the features you want to try.

selected_feature_keys = ['atgcRatio']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(np.stack(train_df["embedding"].values))
test_embeddings = normalize_embeddings(np.stack(test_df["embedding"].values))

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

              precision    recall  f1-score   support

           0       0.92      0.78      0.84     12500
           1       0.81      0.93      0.86     12500

    accuracy                           0.85     25000
   macro avg       0.86      0.85      0.85     25000
weighted avg       0.86      0.85      0.85     25000


🕒 Evaluation took 51.05 seconds (0.85 minutes)


In [13]:
import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)

# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)

}
import time

# Start timer
start_time = time.time()
# Choose the features you want to try.

selected_feature_keys = ['atgcRatio']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(np.stack(train_df["embedding"].values))
test_embeddings = normalize_embeddings(np.stack(test_df["embedding"].values))

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

              precision    recall  f1-score   support

           0       0.92      0.78      0.84     12500
           1       0.81      0.93      0.86     12500

    accuracy                           0.85     25000
   macro avg       0.86      0.85      0.85     25000
weighted avg       0.86      0.85      0.85     25000


🕒 Evaluation took 51.38 seconds (0.86 minutes)


In [14]:
import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)
import time

# Start timer
start_time = time.time()
# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)

}

# Choose the features you want to try.

selected_feature_keys = ['cumulativeSkew']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(np.stack(train_df["embedding"].values))
test_embeddings = normalize_embeddings(np.stack(test_df["embedding"].values))

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

              precision    recall  f1-score   support

           0       0.92      0.78      0.84     12500
           1       0.81      0.93      0.86     12500

    accuracy                           0.85     25000
   macro avg       0.86      0.85      0.85     25000
weighted avg       0.86      0.85      0.85     25000


🕒 Evaluation took 51.06 seconds (0.85 minutes)


In [15]:
import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)
import time

# Start timer
start_time = time.time()
# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)

}

# Choose the features you want to try.

selected_feature_keys = ['pseudoKNC']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(np.stack(train_df["embedding"].values))
test_embeddings = normalize_embeddings(np.stack(test_df["embedding"].values))

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

              precision    recall  f1-score   support

           0       0.92      0.78      0.84     12500
           1       0.81      0.93      0.87     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.86     25000
weighted avg       0.86      0.86      0.86     25000


🕒 Evaluation took 56.38 seconds (0.94 minutes)


In [16]:
train_df.to_csv(r"D:\genomic benchmark\demo_coding_vs_intergenomic_seqs\neucliotide transformer/democodingvsintergenomicseqstrain.csv")

In [17]:
test_df.to_csv(r"D:\genomic benchmark\demo_coding_vs_intergenomic_seqs\neucliotide transformer/democodingvsintergenomicseqstest.csv")